In [1]:
import os
import numpy as np
from obspy import read
from scipy.signal import butter, filtfilt

In [2]:
def channel_to_component(channel):
    channel = channel.upper()
    if channel.endswith("E"):
        return "E"
    if channel.endswith("N"):
        return "N"
    if channel.endswith("Z"):
        return "Z"
    return None

In [3]:
def group_files_by_station(root_folder):
    stations = {}

    for root, _, files in os.walk(root_folder):
        for f in files:
            if not f.endswith(".mseed"):
                continue

            parts = f.split(".")
            if len(parts) < 4:
                continue

            station = parts[1]        # HEF
            channel = parts[3].split("__")[0]  # HHE
            comp = channel_to_component(channel)
            if comp is None:
                continue

            stations.setdefault(station, {"E": [], "N": [], "Z": []})
            stations[station][comp].append(os.path.join(root, f))

    return stations

In [4]:
def load_stream_from_station(station_files):
    components = {}

    for comp in ["E", "N", "Z"]:
        traces = []
        for f in station_files[comp]:
            tr = read(f)[0]
            traces.append(tr.data.astype(float))

        if len(traces) == 0:
            raise ValueError(f"Missing {comp}")

        data = np.concatenate(traces)
        components[comp] = data

    # Trim to shortest length
    min_len = min(len(components[c]) for c in components)
    for c in components:
        components[c] = components[c][:min_len]

    return np.stack(
        [components["E"], components["N"], components["Z"]],
        axis=0
    )

In [5]:
def compute_sta_lta(x, sta, lta):
    if lta <= sta:
        raise ValueError("LTA window must be larger than STA window")

    energy = x**2

    sta_sum = np.convolve(energy, np.ones(sta), "valid")
    lta_sum = np.convolve(energy, np.ones(lta), "valid")

    # Align STA to LTA
    sta_sum = sta_sum[lta - sta:]

    lta_sum[lta_sum == 0] = 1e-12

    ratio = (sta_sum / sta) / (lta_sum / lta)

    print("STA samples:", sta, "LTA samples:", lta)
    print("STA/LTA max:", ratio.max())

    # Pad to original length
    pad = lta - 1
    return np.pad(ratio, (pad, 0), mode="constant")

In [6]:
def bandpass(x, fs, fmin, fmax):
    b, a = butter(4, [fmin/(fs/2), fmax/(fs/2)], btype="band")
    return filtfilt(b, a, x)

In [7]:
def replica_model(stream, detectors, fs=100.0):
    results = []

    for det in detectors:
        delta = det["Delta"]
        sta = int(det["L_wind"] * delta * fs)
        lta = int(det["Sigma"] * delta * fs)
        ndmin = det["NDMIN"]
        thr = det["Thresh_2"]
        fmin, fmax = det["Freq"]

        # Filter each component separately
        filtered = np.array([
            bandpass(stream[i], fs, fmin, fmax)
            for i in range(3)
        ])

        # Compute STA/LTA per component (NO averaging)
        ratios = np.array([
            compute_sta_lta(filtered[i], sta, lta)
            for i in range(3)
        ])

        # Detection if ANY component exceeds threshold
        trigger = np.any(ratios > thr, axis=0)

        # NDMIN logic
        detections = []
        count = 0
        for i, t in enumerate(trigger):
            if t:
                count += 1
                if count == ndmin:
                    detections.append(i)
            else:
                count = 0

        results.append(detections)

    return results

In [8]:
detectors = [
    {"Delta":0.280, "L_wind":4, "Sigma":6, "Thresh_2":2.50, "NDMIN":6,  "Freq":[2.0,6.0]},
    {"Delta":0.100, "L_wind":4, "Sigma":6, "Thresh_2":2.50, "NDMIN":15, "Freq":[5.0,15.0]},
    {"Delta":0.140, "L_wind":4, "Sigma":6, "Thresh_2":2.60, "NDMIN":10, "Freq":[4.0,10.0]},
    {"Delta":0.050, "L_wind":4, "Sigma":6, "Thresh_2":2.60, "NDMIN":15, "Freq":[10.0,35.0]},
]

In [13]:
root = "waveforms_earthquakes_nonoise"
stations = group_files_by_station(root)

valid = {}
for s, files in stations.items():
    try:
        st = load_stream_from_station(files)
        valid[s] = files
    except Exception:
        continue

print("Valid stations:", list(valid.keys())[:])

for station in list(valid.keys())[:]:
    print(f"\nProcessing station {station}")
    st = load_stream_from_station(valid[station])
    detections = replica_model(st, detectors)

    for i, det in enumerate(detections):
        print(f"Detector {i+1}: {len(det)} detections")

Valid stations: ['KIF', 'KMNF', 'KEF', 'KY17', 'VJF', 'LOVF', 'HEF', 'VAF', 'OUL', 'TOF', 'KAF', 'KU2', 'RNF', 'KUNI', 'NUR', 'KY02', 'OBF3', 'PVF', 'OUF', 'SGF', 'KU6', 'KPF', 'AAL', 'RANF', 'OLKF', 'FIA0', 'FIA1', 'HEL5', 'KOFF', 'KU1', 'RSUO', 'MEF', 'RAJF', 'ALAJF', 'KLF', 'OBF8', 'VRF', 'OBF0', 'NIF', 'TVF', 'ECKF', 'RAF', 'TRE03', 'SUF', 'RMF', 'KJNF', 'VUOS', 'MSF', 'LAUT', 'KORF', 'KEV', 'RUF', 'HEL1', 'JOF']

Processing station KIF
STA samples: 112 LTA samples: 168
STA/LTA max: 1.499702160945731
STA samples: 112 LTA samples: 168
STA/LTA max: 1.499742277052791
STA samples: 112 LTA samples: 168
STA/LTA max: 1.4997519905223429
STA samples: 40 LTA samples: 60
STA/LTA max: 1.4989612945810142
STA samples: 40 LTA samples: 60
STA/LTA max: 1.4992653135141343
STA samples: 40 LTA samples: 60
STA/LTA max: 1.4992116699837272
STA samples: 56 LTA samples: 84
STA/LTA max: 1.4996915952005312
STA samples: 56 LTA samples: 84
STA/LTA max: 1.4997005312058942
STA samples: 56 LTA samples: 84
STA/LTA

STA samples: 112 LTA samples: 168
STA/LTA max: 1.4999474231680245
STA samples: 112 LTA samples: 168
STA/LTA max: 1.4997163045943744
STA samples: 112 LTA samples: 168
STA/LTA max: 1.4999603096702545
STA samples: 40 LTA samples: 60
STA/LTA max: 1.499410499959809
STA samples: 40 LTA samples: 60
STA/LTA max: 1.4999625498123055
STA samples: 40 LTA samples: 60
STA/LTA max: 1.4993595313694847
STA samples: 56 LTA samples: 84
STA/LTA max: 1.499663674524365
STA samples: 56 LTA samples: 84
STA/LTA max: 1.4999432298974382
STA samples: 56 LTA samples: 84
STA/LTA max: 1.4996088855447733
STA samples: 20 LTA samples: 30
STA/LTA max: 1.4999772281529233
STA samples: 20 LTA samples: 30
STA/LTA max: 1.4999167464708012
STA samples: 20 LTA samples: 30
STA/LTA max: 1.4999786867385008
Detector 1: 0 detections
Detector 2: 0 detections
Detector 3: 0 detections
Detector 4: 0 detections

Processing station KAF
STA samples: 112 LTA samples: 168
STA/LTA max: 1.499470980149549
STA samples: 112 LTA samples: 168
STA/L

STA samples: 20 LTA samples: 30
STA/LTA max: 1.4997904011778365
STA samples: 20 LTA samples: 30
STA/LTA max: 1.4997661282168298
STA samples: 20 LTA samples: 30
STA/LTA max: 1.4999230321891637
Detector 1: 0 detections
Detector 2: 0 detections
Detector 3: 0 detections
Detector 4: 0 detections

Processing station SGF
STA samples: 112 LTA samples: 168
STA/LTA max: 1.499930073921841
STA samples: 112 LTA samples: 168
STA/LTA max: 1.4996867510925498
STA samples: 112 LTA samples: 168
STA/LTA max: 1.4999159786116574
STA samples: 40 LTA samples: 60
STA/LTA max: 1.4998339500747466
STA samples: 40 LTA samples: 60
STA/LTA max: 1.499787187376053
STA samples: 40 LTA samples: 60
STA/LTA max: 1.4999711113697296
STA samples: 56 LTA samples: 84
STA/LTA max: 1.4997297214085683
STA samples: 56 LTA samples: 84
STA/LTA max: 1.4998637826080976
STA samples: 56 LTA samples: 84
STA/LTA max: 1.4997372040859642
STA samples: 20 LTA samples: 30
STA/LTA max: 1.4999867200860024
STA samples: 20 LTA samples: 30
STA/LTA 

STA samples: 112 LTA samples: 168
STA/LTA max: 1.499883672001155
STA samples: 112 LTA samples: 168
STA/LTA max: 1.4995919862875509
STA samples: 112 LTA samples: 168
STA/LTA max: 1.4999147443542433
STA samples: 40 LTA samples: 60
STA/LTA max: 1.4997884292395744
STA samples: 40 LTA samples: 60
STA/LTA max: 1.499797449848668
STA samples: 40 LTA samples: 60
STA/LTA max: 1.4997843836531246
STA samples: 56 LTA samples: 84
STA/LTA max: 1.4997187283057989
STA samples: 56 LTA samples: 84
STA/LTA max: 1.4998655490841182
STA samples: 56 LTA samples: 84
STA/LTA max: 1.4996631288526072
STA samples: 20 LTA samples: 30
STA/LTA max: 1.499946558004024
STA samples: 20 LTA samples: 30
STA/LTA max: 1.4999131311542522
STA samples: 20 LTA samples: 30
STA/LTA max: 1.4999455896336735
Detector 1: 0 detections
Detector 2: 0 detections
Detector 3: 0 detections
Detector 4: 0 detections

Processing station KU1
STA samples: 112 LTA samples: 168
STA/LTA max: 1.499452457371249
STA samples: 112 LTA samples: 168
STA/LT

STA samples: 112 LTA samples: 168
STA/LTA max: 1.4998357059599976
STA samples: 40 LTA samples: 60
STA/LTA max: 1.499481887722273
STA samples: 40 LTA samples: 60
STA/LTA max: 1.4998473215430022
STA samples: 40 LTA samples: 60
STA/LTA max: 1.499574700077882
STA samples: 56 LTA samples: 84
STA/LTA max: 1.499537687812059
STA samples: 56 LTA samples: 84
STA/LTA max: 1.4997506776574887
STA samples: 56 LTA samples: 84
STA/LTA max: 1.499607719793534
STA samples: 20 LTA samples: 30
STA/LTA max: 1.4997739872028215
STA samples: 20 LTA samples: 30
STA/LTA max: 1.4998561379416187
STA samples: 20 LTA samples: 30
STA/LTA max: 1.4999247268112184
Detector 1: 0 detections
Detector 2: 0 detections
Detector 3: 0 detections
Detector 4: 0 detections

Processing station NIF
STA samples: 112 LTA samples: 168
STA/LTA max: 1.4988866898704774
STA samples: 112 LTA samples: 168
STA/LTA max: 1.4984040371921261
STA samples: 112 LTA samples: 168
STA/LTA max: 1.499434114184458
STA samples: 40 LTA samples: 60
STA/LTA m

STA samples: 20 LTA samples: 30
STA/LTA max: 1.4985008635798596
STA samples: 20 LTA samples: 30
STA/LTA max: 1.4996306088385516
STA samples: 20 LTA samples: 30
STA/LTA max: 1.4996730715381539
Detector 1: 0 detections
Detector 2: 0 detections
Detector 3: 0 detections
Detector 4: 0 detections

Processing station MSF
STA samples: 112 LTA samples: 168
STA/LTA max: 1.4995862583452175
STA samples: 112 LTA samples: 168
STA/LTA max: 1.4999371666608636
STA samples: 112 LTA samples: 168
STA/LTA max: 1.4996644478325134
STA samples: 40 LTA samples: 60
STA/LTA max: 1.4997687874803438
STA samples: 40 LTA samples: 60
STA/LTA max: 1.4998330440114425
STA samples: 40 LTA samples: 60
STA/LTA max: 1.499893345791213
STA samples: 56 LTA samples: 84
STA/LTA max: 1.4998219940564808
STA samples: 56 LTA samples: 84
STA/LTA max: 1.499841305343956
STA samples: 56 LTA samples: 84
STA/LTA max: 1.4997857231278042
STA samples: 20 LTA samples: 30
STA/LTA max: 1.4999173683229634
STA samples: 20 LTA samples: 30
STA/LTA 

In [12]:
# chronogical one
def load_stream_from_station(station_files):
    """
    Load all mseed files for a single station, grouped by component (E, N, Z),
    sort them chronologically by start time, and concatenate into continuous streams.

    station_files: dict of component -> list of file paths
        Example:
        {
            'E': ['file1.mseed', 'file2.mseed'],
            'N': ['file1.mseed', 'file2.mseed'],
            'Z': ['file1.mseed', 'file2.mseed']
        }

    Returns:
        np.array of shape (3, n_samples) with order [E, N, Z]
    """

    components = {}
    for comp in ['E', 'N', 'Z']:
        if comp not in station_files or len(station_files[comp]) == 0:
            raise ValueError(f"Missing component {comp} in station")
        
        # Sort files by start time
        sorted_files = sorted(
            station_files[comp],
            key=lambda f: read(f)[0].stats.starttime
        )

        # Concatenate traces for this component
        traces = [read(f)[0].data for f in sorted_files]
        components[comp] = np.concatenate(traces)

    # Make sure all components have the same length
    min_len = min(len(components['E']), len(components['N']), len(components['Z']))
    stream_array = np.stack([
        components['E'][:min_len],
        components['N'][:min_len],
        components['Z'][:min_len]
    ], axis=0)

    return stream_array